### Task 1: Data Preparation (with scikit-learn)

In [59]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
print("Project root added:", project_root)


Project root added: /Users/mac/Desktop/nlp-learner


In [2]:
texts = [
    "This movie is fantastic and I love it!",
    "I hate this film, it's terrible.",
    "The acting was superb, a truly great experience.",
    "What a waste of time, absolutely boring.",
    "Highly recommend this, a masterpiece.",
    "Could not finish watching, so bad."
]
labels = [1, 0, 1, 0, 1, 0]

In [3]:
train_texts = texts[:4]
train_labels = labels[:4]
test_texts = texts[4:]
test_labels = labels[4:]

### Logic building 

In [10]:
from src.preprocessing.simple_tokenizer import SimpleTokenizer
from src.representations.count_vectorizer import CountVectorizer
from src.representations.tfidf_vectorizer import TfidfVectorizerCustom

tokenizer = SimpleTokenizer()
count_vectorizer = CountVectorizer(tokenizer=tokenizer)
tfidf_vectorizer = TfidfVectorizerCustom(count_vectorizer)
tfidfMatrix = tfidf_vectorizer.fit_transform(texts)



### Result Matrix

In [5]:
import pandas as pd

vocab = tfidf_vectorizer.count_vectorizer.vocabulary_

sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
columns = [word for word, _ in sorted_vocab]

df = pd.DataFrame(tfidfMatrix, columns=columns)
print(df.shape)
print(df.head())


(6, 35)
          a  absolutely    acting       and  bad    boring  could  experience  \
0  0.000000    0.000000  0.000000  2.252763  0.0  0.000000    0.0    0.000000   
1  0.000000    0.000000  0.000000  0.000000  0.0  0.000000    0.0    0.000000   
2  1.559616    0.000000  2.252763  0.000000  0.0  0.000000    0.0    2.252763   
3  1.559616    2.252763  0.000000  0.000000  0.0  2.252763    0.0    0.000000   
4  1.559616    0.000000  0.000000  0.000000  0.0  0.000000    0.0    0.000000   

   fantastic      film  ...    superb  terrible       the      this      time  \
0   2.252763  0.000000  ...  0.000000  0.000000  0.000000  1.559616  0.000000   
1   0.000000  2.252763  ...  0.000000  2.252763  0.000000  1.559616  0.000000   
2   0.000000  0.000000  ...  2.252763  0.000000  2.252763  0.000000  0.000000   
3   0.000000  0.000000  ...  0.000000  0.000000  0.000000  0.000000  2.252763   
4   0.000000  0.000000  ...  0.000000  0.000000  0.000000  1.559616  0.000000   

      truly       

### Using Library

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from src.core.text_classifier import TextClassifier
import pandas as pd
# Create and train 
vectorizer = TfidfVectorizer()




In [7]:
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X = vectorizer.fit_transform(texts)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("TF-IDF matrix shape:", X.shape)

df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(df)

Vocabulary size: 21
TF-IDF matrix shape: (6, 21)
   absolutely    acting      bad  boring  experience  fantastic     film  \
0         0.0  0.000000  0.00000     0.0    0.000000    0.57735  0.00000   
1         0.0  0.000000  0.00000     0.0    0.000000    0.00000  0.57735   
2         0.0  0.447214  0.00000     0.0    0.447214    0.00000  0.00000   
3         0.5  0.000000  0.00000     0.5    0.000000    0.00000  0.00000   
4         0.0  0.000000  0.00000     0.0    0.000000    0.00000  0.00000   
5         0.0  0.000000  0.57735     0.0    0.000000    0.00000  0.00000   

    finish     great     hate  ...     love  masterpiece    movie  recommend  \
0  0.00000  0.000000  0.00000  ...  0.57735      0.00000  0.57735    0.00000   
1  0.00000  0.000000  0.57735  ...  0.00000      0.00000  0.00000    0.00000   
2  0.00000  0.447214  0.00000  ...  0.00000      0.00000  0.00000    0.00000   
3  0.00000  0.000000  0.00000  ...  0.00000      0.00000  0.00000    0.00000   
4  0.00000  0.0000

### Task 2: TextClassifier Implementation

In [8]:
classifier = TextClassifier(vectorizer)
classifier.fit(train_texts, train_labels)
preds = classifier.predict(texts)

metrics = classifier.evaluate(labels, preds)

print("Predictions:", preds)
print("Metrics:", metrics)

Predictions: [1, 0, 1, 0, 0, 0]
Metrics: {'accuracy': 0.8333333333333334, 'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8}


### Task 3: Evaluation

In [20]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from src.preprocessing.regex_tokenizer import RegexTokenizer


X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Initialize the tokenizer and vectorizer
tokenizer = RegexTokenizer()
count_vectorizer = CountVectorizer(tokenizer=tokenizer)
tfidf_vectorizer = TfidfVectorizerCustom(count_vectorizer)

# Instantiate and train classifier
classifier = TextClassifier(vectorizer)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)


Accuracy: 0.5


In [21]:
print("Classification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/Users/mac/Desktop/nlp-learner/venv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Desktop/nlp-learner/venv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Desktop/nlp-learner/venv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

In [19]:
tfidfMatrix = tfidf_vectorizer.fit_transform(texts)
vocab = tfidf_vectorizer.count_vectorizer.vocabulary_

sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])
columns = [word for word, _ in sorted_vocab]

df = pd.DataFrame(tfidfMatrix, columns=columns)
print(df.shape)
print(df.head())


(6, 39)
          !         '         ,         .         a  absolutely    acting  \
0  2.252763  0.000000  0.000000  0.000000  0.000000    0.000000  0.000000   
1  0.000000  2.252763  1.154151  1.154151  0.000000    0.000000  0.000000   
2  0.000000  0.000000  1.154151  1.154151  1.559616    0.000000  2.252763   
3  0.000000  0.000000  1.154151  1.154151  1.559616    2.252763  0.000000   
4  0.000000  0.000000  1.154151  1.154151  1.559616    0.000000  0.000000   

        and  bad    boring  ...    superb  terrible       the      this  \
0  2.252763  0.0  0.000000  ...  0.000000  0.000000  0.000000  1.559616   
1  0.000000  0.0  0.000000  ...  0.000000  2.252763  0.000000  1.559616   
2  0.000000  0.0  0.000000  ...  2.252763  0.000000  2.252763  0.000000   
3  0.000000  0.0  2.252763  ...  0.000000  0.000000  0.000000  0.000000   
4  0.000000  0.0  0.000000  ...  0.000000  0.000000  0.000000  1.559616   

       time     truly       was     waste  watching      what  
0  0.000000  0

### Task 4 Advanced Example: Sentiment Analysis with PySpark

In [87]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SentimentAnalysis").getOrCreate()


In [88]:
from pyspark.sql.functions import col

data_path = "sentiments.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

# Convert -1/1 labels to 0/1: Normalize sentiment labels
df = df.withColumn("label", (col("sentiment").cast("integer") + 1) / 2)

# Drop rows with null sentiment
df = df.dropna(subset=["sentiment"])


#### Preprocessing Pipeline

In [34]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

# Tokenize text
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Remove stop words
stopwordsRemover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# Convert tokens → feature vector
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=10000)

# Rescale with IDF
idf = IDF(inputCol="raw_features", outputCol="features")


25/11/10 23:21:58 WARN StopWordsRemover: Default locale set was [en_CN]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.


#### Train the LogisticRegression Model

In [35]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    maxIter=10, 
    regParam=0.001, 
    featuresCol="features", 
    labelCol="label"
)

pipeline = Pipeline(stages=[tokenizer, stopwordsRemover, hashingTF, idf, lr])


In [36]:
trainingData, testData = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(trainingData)


25/11/10 23:22:37 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [37]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions = model.transform(testData)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy: {accuracy}")


Accuracy: 0.7294860234445446


#### Improve More Model Performance

In [89]:
from typing import List
from notebook.newTokenizer import newTokenizer
from pyspark.sql.functions import col
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StopWordsRemover, HashingTF, IDF, Word2Vec
from pyspark.ml.classification import LogisticRegression, NaiveBayes, GBTClassifier, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import udf



In [90]:
class newTokenizer:
    def tokenizer(self, text: str):
        if text is None:
            return []
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", "", text)
        text = re.sub(r"<.*?>", "", text)
        text = re.sub(r"[^a-z0-9\s]", "", text)
        return text.split()

tokenizer = newTokenizer()
tokenizer_udf = udf(tokenizer.tokenizer, ArrayType(StringType()))
df = df.withColumn("filtered_words", tokenizer_udf(col("text")))

In [91]:
stopwords_remover = StopWordsRemover(inputCol="filtered_words", outputCol="final_words")
df = stopwords_remover.transform(df)


trainData, testData = df.randomSplit([0.8, 0.2], seed=42)


25/11/10 23:54:20 WARN StopWordsRemover: Default locale set was [en_CN]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.


In [92]:
# TF-IDF
hashingTF = HashingTF(inputCol="final_words", outputCol="raw_features", numFeatures=5000)
idf = IDF(inputCol="raw_features", outputCol="features_tfidf")

# Word2Vec
word2vec = Word2Vec(inputCol="final_words", outputCol="features_w2v", vectorSize=100)


In [93]:
feature_pipelines = {
    "TF-IDF": ["final_words", hashingTF, idf, "features_tfidf"],
    "Word2Vec": ["final_words", word2vec, None, "features_w2v"]  # IDF not needed for W2V
}

#### Model

In [79]:
models = {
    "LogisticRegression": LogisticRegression(maxIter=10, regParam=0.001),
    "NaiveBayes": NaiveBayes(),
    "GBT": GBTClassifier(maxIter=20),
    "MLP": MultilayerPerceptronClassifier(layers=[5000, 100, 2], maxIter=100)  # only TF-IDF input
}


#### TF-IDF + Model

In [94]:

results = {}

for feat_name, (input_col, feat_stage1, feat_stage2, features_col) in feature_pipelines.items():
    for model_name, model in models.items():
        # Skip incompatible combinations
        if feat_name == "Word2Vec" and model_name in ["NaiveBayes", "GBT", "MLP"]:
            continue  

        model.setParams(featuresCol=features_col, labelCol="label")

        # Build pipeline
        stages = [feat_stage1]
        if feat_stage2:
            stages.append(feat_stage2)
        stages.append(model)

        pipeline = Pipeline(stages=stages)
        pipeline_model = pipeline.fit(trainData)
        predictions = pipeline_model.transform(testData)

        # Evaluate
        evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
        acc = evaluator.evaluate(predictions)

        results[f"{feat_name} + {model_name}"] = acc

# Print results
for k, v in results.items():
    print(f"{k} Accuracy: {v:.4f}")

25/11/10 23:55:55 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


TF-IDF + LogisticRegression Accuracy: 0.7096
TF-IDF + NaiveBayes Accuracy: 0.6916
TF-IDF + GBT Accuracy: 0.7457
TF-IDF + MLP Accuracy: 0.7547
Word2Vec + LogisticRegression Accuracy: 0.6664
